In [2]:
import os

file_path = "../data/processed/feature_dataset.csv"

print("Current folder:", os.getcwd())
print("File location:", os.path.abspath(file_path))
print("File size in GB:", round(os.path.getsize(file_path) / (1024**3), 2))

Current folder: c:\AI-Energy-Optimization-Platform\notebooks
File location: c:\AI-Energy-Optimization-Platform\data\processed\feature_dataset.csv
File size in GB: 1.12


In [2]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/feature_dataset.csv",
    parse_dates=["day"],
    low_memory=False
)

print(df.shape)

(3302637, 45)


In [3]:
from sklearn.model_selection import train_test_split

In [4]:
TARGET = "energy_sum"

X = df.drop(columns=[TARGET])

y = df[TARGET]

In [5]:
X.dtypes.value_counts()

float64           26
int64              9
object             8
datetime64[ns]     1
Name: count, dtype: int64

In [6]:
print("Object columns:")
print(X.select_dtypes(include="object").columns.tolist())

print("\nDatetime columns:")
print(X.select_dtypes(include="datetime").columns.tolist())

Object columns:
['LCLid', 'stdorToU', 'Acorn', 'Acorn_grouped', 'icon', 'precipType', 'summary', 'season']

Datetime columns:
['day']


In [7]:
X = X.drop(columns=["LCLid", "day"])

In [8]:
categorical_cols = [
    "stdorToU",
    "Acorn",
    "Acorn_grouped",
    "icon",
    "precipType",
    "summary",
    "season"
]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

In [9]:
print(X.shape)
print(X.dtypes.value_counts())

(3302637, 150)
bool       115
float64     26
int64        9
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(2642109, 150)
(660528, 150)


In [11]:
from xgboost import XGBRegressor

In [12]:
model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [13]:
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 0.14086220263331975
RMSE: 0.7591526654494077
R²  : 0.9931564952467471


In [15]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
1,energy_mean,0.358003
0,energy_median,0.221152
33,rolling_mean_7,0.108389
30,lag_1,0.043559
28,quarter,0.033706
4,energy_std,0.026411
2,energy_max,0.012515
39,Acorn_ACORN-D,0.012368
5,energy_min,0.010289
43,Acorn_ACORN-H,0.009615


In [16]:
import joblib

joblib.dump(model, "../models/energy_forecast_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [17]:
joblib.dump(X.columns.tolist(), "../models/features.pkl")

['../models/features.pkl']